[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/templates/43_masked_pairwise_dist.ipynb)

# 🟡 Medium: Masked Pairwise Distance

Given `points` of shape `(N, D)` and `group_ids` of shape `(N,)`, return a matrix of shape `(N, N)` where:

- `D[i, j] = ||points[i] - points[j]||²` if `group_ids[i] == group_ids[j]`
- `D[i, j] = float('inf')` otherwise

This is the same broadcasting pattern as masked attention, and shows up in **supervised contrastive learning** (only compare within class) and **graph neural networks** (only compare within neighborhood).

### Signature
```python
def masked_pairwise_dist(points: torch.Tensor, group_ids: torch.Tensor) -> torch.Tensor:
    ...
```

### Rules
- Do **NOT** use Python `for` loops
- Derive the mask from `group_ids[:, None] == group_ids[None, :]`

### Example
```
points    = [[0,0],[1,0],[0,1],[10,10]]   group_ids = [0, 0, 1, 0]

output:
[[  0,   1, inf, 200],
 [  1,   0, inf, 181],
 [inf, inf,   0, inf],
 [200, 181, inf,   0]]
```

> **Reduction step (say this before coding):** `D[i,j]` depends on two independent things: the squared distance between points `i` and `j`, AND whether `group_ids[i] == group_ids[j]`. The second condition is an **outer-product comparison** — not a loop.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def masked_pairwise_dist(points: torch.Tensor, group_ids: torch.Tensor) -> torch.Tensor:
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
points    = torch.tensor([[0.,0.],[1.,0.],[0.,1.],[10.,10.]])
group_ids = torch.tensor([0, 0, 1, 0])
D = masked_pairwise_dist(points, group_ids)
print('Output:')
print(D)
print('Shape:', D.shape, '  (should be (4, 4))')
print('Symmetric:', torch.allclose(D, D.T, equal_nan=False))

In [ ]:
import torch, math, time

# ── Test 1: spec example (4 points, 2 groups) ─────────────────────────────
points    = torch.tensor([[0.,0.],[1.,0.],[0.,1.],[10.,10.]])
group_ids = torch.tensor([0, 0, 1, 0])
D = masked_pairwise_dist(points, group_ids)
assert D.shape == (4, 4), f"Shape: {D.shape}"
assert D[0,1].item() == 1.0,   f"D[0,1]={D[0,1]}"
assert D[1,3].item() == 181.0, f"D[1,3]={D[1,3]}"
assert D[0,3].item() == 200.0, f"D[0,3]={D[0,3]}"
assert math.isinf(D[0,2].item()), f"D[0,2] should be inf, got {D[0,2]}"
assert math.isinf(D[2,0].item()), f"D[2,0] should be inf, got {D[2,0]}"
assert math.isinf(D[2,1].item()), f"D[2,1] should be inf, got {D[2,1]}"
assert math.isinf(D[2,3].item()), f"D[2,3] should be inf, got {D[2,3]}"
print("Test 1 passed: spec example")

# ── Test 2: diagonal is always 0 ──────────────────────────────────────────
torch.manual_seed(42)
points = torch.randn(8, 4)
group_ids = torch.randint(0, 3, (8,))
D = masked_pairwise_dist(points, group_ids)
assert (D.diagonal() == 0).all(), f"Diagonal not all zero: {D.diagonal()}"
print("Test 2 passed: diagonal is zero")

# ── Test 3: output is symmetric ───────────────────────────────────────────
torch.manual_seed(7)
points = torch.randn(10, 6)
group_ids = torch.randint(0, 4, (10,))
D = masked_pairwise_dist(points, group_ids)
assert torch.allclose(D, D.T, equal_nan=False), "Output is not symmetric"
print("Test 3 passed: symmetry")

# ── Test 4: all same group → no inf ───────────────────────────────────────
points = torch.randn(5, 3)
group_ids = torch.zeros(5, dtype=torch.long)
D = masked_pairwise_dist(points, group_ids)
assert not any(math.isinf(D[i,j].item()) for i in range(5) for j in range(5)),     "Should have no inf when all same group"
print("Test 4 passed: all same group has no inf")

# ── Test 5: all distinct groups → only diagonal finite ────────────────────
points = torch.randn(4, 2)
group_ids = torch.tensor([0, 1, 2, 3])
D = masked_pairwise_dist(points, group_ids)
for i in range(4):
    for j in range(4):
        if i == j:
            assert D[i,j].item() == 0.0, f"Diagonal D[{i},{j}] should be 0"
        else:
            assert math.isinf(D[i,j].item()), f"Off-diagonal D[{i},{j}] should be inf"
print("Test 5 passed: all distinct groups")

# ── Test 6: large N=500 D=64 (timing) ─────────────────────────────────────
torch.manual_seed(0)
N, Dim = 500, 64
points = torch.randn(N, Dim)
group_ids = torch.randint(0, 10, (N,))
t0 = time.time()
result = masked_pairwise_dist(points, group_ids)
elapsed = time.time() - t0
assert result.shape == (N, N), f"Shape: {result.shape}"
assert elapsed < 2.0, f"Too slow: {elapsed:.2f}s (expected <2s — no loops)"
print(f"Test 6 passed: N=500 timing ({elapsed:.3f}s)")

print("\nAll tests passed!")
